# Stage 5 - Forecasting the stock-bond correlation

Stage 4 found that risk parity hardly reacts to the macro regimes, but that knowing next month's SPY-TLT
correlation in advance would be worth about 0.2 of Sharpe ratio. This stage asks the supervised version of the question:

1. Can next month's SPY-TLT correlation be forecast with data known today, better than by its recent average?
2. Does macro data add anything to market data?
3. How much of the value of a perfect forecast does a realistic forecast recover in the portfolio?

- **Input:** `data/processed/` (stages 2 to 4) and the market series in `data/raw/` (term and credit spread)
- **Output:** `data/processed/forecast_predictions.csv`, `forecast_accuracy.csv`, `forecast_importance.csv`,
  `forecast_portfolio_returns.csv`, `forecast_performance.csv`

Run from top to bottom in a fresh kernel (about 5 minutes: the models are refitted every month).

In [1]:
# the notebook needs code/ as working directory to import config and helpers:
# VS Code starts there, JupyterLab on Nuvolos starts in /files, so look for the project
import os
from pathlib import Path

if not Path("helpers.py").exists():
    found = [p.parent for p in Path.cwd().glob("**/code/01_get_data.py") if (p.parent / "helpers.py").exists()]
    if len(found) != 1:
        raise FileNotFoundError(f"cannot find a single project code folder under {Path.cwd()}: {found}")
    os.chdir(found[0])
print("working directory:", Path.cwd())

import time

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import config
import helpers

working directory: /Users/marcodamico/Desktop/hec/semestre 3/data science/Advanced_Data_Analytics/code


## 1. Design, fixed before looking at the results

**Target.** The correlation of daily SPY and TLT returns in month `t+1`: the month the portfolio is held.
Models predict its Fisher transform, `arctanh(correlation)`, so that forecasts stay between -1 and 1;
errors are measured on the correlation itself.

**Features**, all known at month-end `t`:

| Group | Feature | Source |
|---|---|---|
| market | correlation this month, over the last 3 months, average of the last 12 monthly correlations | stage 2 |
| market | volatility of SPY and of TLT this month (log, annualised) | stage 2 |
| market | term spread (10y - 2y Treasury) and credit spread (Baa - 10y), last value of the month | FRED |
| macro | PC1 (economic cycle) and PC2 (inflation against growth), real time, expanding window | stage 3 |
| macro | CPI inflation over 12 months, real time | stage 2 |

The Chicago Fed financial conditions index (NFCI) is not used: it is revised and only today's version was
downloaded, so it would use information not known at the time.

**Models.**
- benchmarks: historical mean of the monthly correlation; last month's correlation; the 12-month average
  (the forecast of `rp_rolling` in stage 4, the best simple forecast found there);
- linear regression, ridge regression, random forest, gradient boosting. Ridge, random forest and gradient boosting
  are fitted twice, on market features only and on all features: the difference is the value of macro data.

**Walk-forward.** At each month-end `t` from December 2006 to July 2026 (the 236 decisions of stage 4), each model
is trained on every pair (features at `s`, correlation in `s+1`) already known at `t`, at least 36 of them,
and forecasts month `t+1`. Models are refitted every month. Hyperparameters are chosen by time-series
cross-validation (5 folds, each validated on later months than it is trained on) inside the training data,
once a year in December. The grids are small and fixed here.

**Accuracy.** Root mean squared error, out-of-sample R² against the historical mean and against the 12-month average,
and the Diebold-Mariano test against the 12-month average (squared errors, Newey-West standard errors).

**Portfolio.** The risk parity of stage 4 (same volatilities, 10% target, at most 2x, 10 bps of costs), with the
12-month average correlation for every pair of assets except SPY-TLT, which is the forecast. So against `rp_rolling`
the only difference is the SPY-TLT forecast. Upper bound: `oracle_spy_tlt`, the true SPY-TLT correlation of
next month. Periods as in stage 4.

In [2]:
# portfolio settings: the same as stage 4 (checked below against its output)
ASSETS = ["SPY", "TLT", "GLD", "DBC"]
FIRST_DECISION = "2006-12-31"
CORR_START = "2006-02-01"
VOL_DAYS = 63
ROLLING_MONTHS = 12
TARGET_VOL = 0.10
MAX_LEVERAGE = 2.0
COST_BPS = 10
PERIODS = {"full sample": ("2007", "2026"), "2007-2016": ("2007", "2016"), "2017-2026": ("2017", "2026"),
           "2008": ("2008", "2008"), "2020": ("2020", "2020"), "2022": ("2022", "2022")}

# forecasting settings
MIN_TRAIN = 36
CV_FOLDS = 5
MARKET = ["corr_1m", "corr_3m", "corr_12m", "log_vol_spy", "log_vol_tlt", "term_spread", "credit_spread"]
MACRO = ["pc1", "pc2", "inflation"]
ALL = MARKET + MACRO


def forest():
    return RandomForestRegressor(n_estimators=200, min_samples_leaf=5, max_features=0.5,
                                 random_state=config.RANDOM_SEED)


def boosting():
    return GradientBoostingRegressor(learning_rate=0.05, subsample=0.8, random_state=config.RANDOM_SEED)


MODELS = {   # name -> (estimator, hyperparameter grid, features)
    "ols_all": (make_pipeline(StandardScaler(), LinearRegression()), {}, ALL),
    "ridge_market": (make_pipeline(StandardScaler(), Ridge()), {"ridge__alpha": [0.1, 1, 10, 100]}, MARKET),
    "ridge_all": (make_pipeline(StandardScaler(), Ridge()), {"ridge__alpha": [0.1, 1, 10, 100]}, ALL),
    "forest_market": (forest(), {"max_depth": [2, 3, 5]}, MARKET),
    "forest_all": (forest(), {"max_depth": [2, 3, 5]}, ALL),
    "boosting_market": (boosting(), {"n_estimators": [50, 150], "max_depth": [1, 2]}, MARKET),
    "boosting_all": (boosting(), {"n_estimators": [50, 150], "max_depth": [1, 2]}, ALL),
}
BENCHMARKS = ["historical_mean", "last_month", "rolling_12m"]

## 2. Features and target

In [3]:
def load(name):
    return helpers.load_processed(name, index_col="date", parse_dates=True)


sb = load("stock_bond_corr")["spy_tlt_corr"]
sb_3m = load("stock_bond_corr")["spy_tlt_corr_3m"]
ret_d_all = load("returns_daily")
pca = load("pca_scores")
macro = load("macro_realtime_12m")
market = helpers.load_market_series()


def month_end(series_id):
    """Last daily value of a market series in each month."""
    s = market[market["series_id"] == series_id].set_index("date")["value"].dropna()
    return s.resample("ME").last()


daily = ret_d_all[["SPY", "TLT"]]
days = daily.resample("ME").count()
vol = (daily.resample("ME").std() * np.sqrt(252)).where(days >= 15)

data = pd.DataFrame({
    "corr_1m": sb,
    "corr_3m": sb_3m,
    "corr_12m": sb.rolling(12, min_periods=6).mean(),
    "log_vol_spy": np.log(vol["SPY"]),
    "log_vol_tlt": np.log(vol["TLT"]),
    "term_spread": month_end("T10Y2Y"),
    "credit_spread": month_end("BAA10Y"),
    "pc1": pca["realtime_12m_pc1"],
    "pc2": pca["realtime_12m_pc2"],
    "inflation": macro["cpi"],
}).reindex(sb.index)
data["target"] = sb.shift(-1)            # correlation of month t+1 (index is month-end and every month is present)
assert (sb.index.to_series().diff().dt.days.dropna() <= 31).all()
data = data.dropna(subset=ALL)
print(f"{len(data)} months with all features, {data.index[0].date()} -> {data.index[-1].date()}")
data[ALL + ["target"]].describe().round(2)

284 months with all features, 2003-01-31 -> 2026-08-31


,corr_1m,corr_3m,corr_12m,log_vol_spy,log_vol_tlt,term_spread,credit_spread,pc1,pc2,inflation,target
count,284.00,284.00,284.00,284.00,284.00,284.00,284.00,284.00,284.00,284.00,283.00
mean,-0.24,-0.25,-0.26,-2.02,-2.09,1.05,2.40,0.54,0.16,2.55,-0.23
std,0.38,0.31,0.24,0.50,0.35,0.96,0.76,2.51,1.62,1.73,0.38
min,-0.87,-0.83,-0.72,-3.15,-3.10,-1.06,1.42,-8.00,-3.26,-1.91,-0.87
25%,-0.54,-0.51,-0.44,-2.37,-2.34,0.25,1.85,-0.02,-0.97,1.64,-0.54
50%,-0.30,-0.27,-0.31,-2.06,-2.12,0.96,2.24,0.84,0.13,2.35,-0.30
75%,0.06,-0.01,-0.03,-1.72,-1.85,1.86,2.82,1.71,0.91,3.24,0.06
max,0.87,0.55,0.29,-0.10,-0.51,2.84,6.10,7.79,6.32,8.61,0.87


In [4]:
ret_m = load("returns_monthly")[ASSETS]
decisions = ret_m.loc[FIRST_DECISION:].index[:-1]
assert data.reindex(decisions)[ALL].notna().all().all()
first_train = data.loc[: decisions[0] - pd.offsets.MonthEnd(1)].dropna(subset=["target"])
print(f"{len(decisions)} forecasts, {decisions[0].date()} -> {decisions[-1].date()};"
      f" first training sample: {len(first_train)} months")
print("\ncorrelation between features (full sample, description only):")
print(data[ALL].corr().round(2).to_string())

236 forecasts, 2006-12-31 -> 2026-07-31; first training sample: 47 months

correlation between features (full sample, description only):
               corr_1m  corr_3m  corr_12m  log_vol_spy  log_vol_tlt  term_spread  credit_spread   pc1   pc2  inflation
corr_1m           1.00     0.80      0.63        -0.27        -0.17        -0.37          -0.48  0.08  0.18       0.23
corr_3m           0.80     1.00      0.80        -0.20        -0.14        -0.46          -0.58  0.12  0.25       0.31
corr_12m          0.63     0.80      1.00        -0.23        -0.26        -0.65          -0.73  0.24  0.31       0.41
log_vol_spy      -0.27    -0.20     -0.23         1.00         0.57         0.13           0.56 -0.20  0.27       0.14
log_vol_tlt      -0.17    -0.14     -0.26         0.57         1.00         0.15           0.48 -0.11  0.02       0.10
term_spread      -0.37    -0.46     -0.65         0.13         0.15         1.00           0.55 -0.17 -0.27      -0.43
credit_spread    -0.48    -0.5

## 3. Walk-forward forecasts

In [5]:
def fisher(r):
    return np.arctanh(np.clip(r, -0.99, 0.99))


def importances(model, cols):
    """Share of each feature in the fitted model: |standardised coefficient| for linear models, impurity for trees."""
    last = model[-1] if hasattr(model, "steps") else model
    raw = np.abs(last.coef_) if hasattr(last, "coef_") else last.feature_importances_
    return pd.Series(raw / raw.sum(), index=cols)


forecasts = {name: {} for name in list(MODELS) + BENCHMARKS}
chosen, imp = {}, {name: [] for name in MODELS}
start = time.time()
for t in decisions:
    train = data.loc[: t - pd.offsets.MonthEnd(1)].dropna(subset=["target"])   # targets known at t
    assert len(train) >= MIN_TRAIN and train.index[-1] + pd.offsets.MonthEnd(1) <= t
    z = fisher(train["target"])
    for name, (estimator, grid, cols) in MODELS.items():
        if grid and (name not in chosen or t.month == 12):
            search = GridSearchCV(estimator, grid, cv=TimeSeriesSplit(CV_FOLDS),
                                  scoring="neg_mean_squared_error").fit(train[cols], z)
            chosen[name] = search.best_params_
        model = clone(estimator).set_params(**chosen.get(name, {})).fit(train[cols], z)
        forecasts[name][t] = float(np.tanh(model.predict(data.loc[[t], cols])[0]))
        imp[name].append(importances(model, cols))
    forecasts["historical_mean"][t] = sb.loc[:t].mean()
    forecasts["last_month"][t] = data.loc[t, "corr_1m"]
print(f"done in {time.time() - start:.0f} s")
print("hyperparameters chosen in the last December:", chosen)

done in 325 s
hyperparameters chosen in the last December: {'ridge_market': {'ridge__alpha': 100}, 'ridge_all': {'ridge__alpha': 100}, 'forest_market': {'max_depth': 2}, 'forest_all': {'max_depth': 2}, 'boosting_market': {'max_depth': 1, 'n_estimators': 50}, 'boosting_all': {'max_depth': 1, 'n_estimators': 50}}


The 12-month average is taken from the same monthly correlation matrices as stage 4, so that the
`rolling_12m` portfolio below is exactly `rp_rolling`.

In [6]:
ret_d = ret_d_all[ASSETS].loc[CORR_START:]
corr_months, C = helpers.monthly_correlations(ret_d)


def rolling_matrix(t):
    """Average of the last 12 monthly correlation matrices known at t (as rp_rolling in stage 4)."""
    return C[corr_months <= t][-ROLLING_MONTHS:].mean(axis=0)


forecasts["rolling_12m"] = {t: rolling_matrix(t)[0, 1] for t in decisions}
forecasts = pd.DataFrame(forecasts)[BENCHMARKS + list(MODELS)].rename_axis("date")
realised = data["target"].reindex(decisions)
assert realised.notna().all()
forecasts.describe().round(2)

,historical_mean,last_month,rolling_12m,ols_all,ridge_market,ridge_all,forest_market,forest_all,boosting_market,boosting_all
count,236.00,236.00,236.00,236.00,236.00,236.00,236.00,236.00,236.00,236.00
mean,-0.29,-0.27,-0.28,-0.28,-0.28,-0.28,-0.30,-0.30,-0.30,-0.30
std,0.06,0.37,0.24,0.27,0.20,0.20,0.21,0.20,0.19,0.19
min,-0.35,-0.87,-0.72,-0.97,-0.65,-0.67,-0.68,-0.70,-0.65,-0.73
25%,-0.34,-0.56,-0.45,-0.47,-0.43,-0.42,-0.48,-0.47,-0.46,-0.46
50%,-0.32,-0.36,-0.34,-0.32,-0.32,-0.31,-0.33,-0.33,-0.31,-0.31
75%,-0.27,-0.03,-0.11,-0.09,-0.16,-0.15,-0.16,-0.15,-0.15,-0.16
max,-0.12,0.87,0.29,0.38,0.30,0.31,0.16,0.14,0.23,0.15


## 4. Accuracy

In [7]:
def diebold_mariano(e_model, e_bench):
    """Mean difference of squared errors (model - benchmark) and its Newey-West p-value; negative = model better."""
    d = (e_model ** 2 - e_bench ** 2).rename("loss_difference")
    table, _ = helpers.ols_hac(d, pd.DataFrame(index=d.index))
    return table.loc["const", "coef"], table.loc["const", "p"]


errors = forecasts.sub(realised, axis=0)
rows = {}
for period, (a, b) in PERIODS.items():
    e = errors.loc[a:b]
    for name in forecasts:
        dm, p = diebold_mariano(e[name], e["rolling_12m"]) if name != "rolling_12m" else (0.0, np.nan)
        rows[(period, name)] = {
            "rmse": np.sqrt((e[name] ** 2).mean()),
            "r2 vs mean": 1 - (e[name] ** 2).sum() / (e["historical_mean"] ** 2).sum(),
            "r2 vs rolling": 1 - (e[name] ** 2).sum() / (e["rolling_12m"] ** 2).sum(),
            "dm vs rolling": dm, "p": p, "months": len(e)}
accuracy = pd.DataFrame(rows).T
accuracy.index.names = ["period", "model"]
accuracy.loc["full sample"].round(3)

,rmse,r2 vs mean,r2 vs rolling,dm vs rolling,p,months
model,,,,,,
historical_mean,0.378,0.000,-0.470,0.046,0.002,235.0
last_month,0.364,0.075,-0.360,0.035,0.003,235.0
rolling_12m,0.312,0.320,0.000,0.000,NaN,235.0
ols_all,0.321,0.282,-0.055,0.005,0.480,235.0
ridge_market,0.307,0.342,0.033,-0.003,0.537,235.0
ridge_all,0.307,0.343,0.034,-0.003,0.551,235.0
forest_market,0.305,0.352,0.048,-0.005,0.420,235.0
forest_all,0.303,0.358,0.056,-0.005,0.322,235.0
boosting_market,0.315,0.308,-0.017,0.002,0.789,235.0


In [8]:
for period in PERIODS:
    if period != "full sample":
        print(f"\n{period}:")
        print(accuracy.loc[period, ["rmse", "r2 vs rolling", "p"]].round(3).to_string())


2007-2016:
                  rmse  r2 vs rolling      p
model                                       
historical_mean  0.337         -0.220  0.152
last_month       0.335         -0.202  0.257
rolling_12m      0.305          0.000    NaN
ols_all          0.308         -0.016  0.894
ridge_market     0.287          0.116  0.149
ridge_all        0.287          0.115  0.145
forest_market    0.286          0.122  0.181
forest_all       0.284          0.135  0.102
boosting_market  0.297          0.055  0.575
boosting_all     0.289          0.103  0.299

2017-2026:
                  rmse  r2 vs rolling      p
model                                       
historical_mean  0.417         -0.709  0.003
last_month       0.392         -0.511  0.001
rolling_12m      0.319          0.000    NaN
ols_all          0.333         -0.092  0.354
ridge_market     0.326         -0.045  0.513
ridge_all        0.326         -0.043  0.586
forest_market    0.323         -0.023  0.754
forest_all       0.322         

### What the models use

Average share of each feature over the 236 refits.

In [9]:
importance = pd.DataFrame({name: pd.concat(v, axis=1).mean(axis=1) for name, v in imp.items()}).reindex(ALL)
importance.loc["macro, total"] = importance.loc[MACRO].sum()
importance.round(2)

,ols_all,ridge_market,ridge_all,forest_market,forest_all,boosting_market,boosting_all
corr_1m,0.12,0.25,0.19,0.27,0.23,0.33,0.31
corr_3m,0.09,0.22,0.16,0.21,0.20,0.22,0.20
corr_12m,0.08,0.15,0.11,0.19,0.17,0.15,0.14
log_vol_spy,0.09,0.13,0.11,0.06,0.05,0.03,0.03
log_vol_tlt,0.04,0.09,0.07,0.04,0.03,0.03,0.03
term_spread,0.06,0.06,0.05,0.05,0.04,0.03,0.03
credit_spread,0.09,0.09,0.10,0.19,0.16,0.21,0.20
pc1,0.08,NaN,0.08,NaN,0.05,NaN,0.03
pc2,0.16,NaN,0.07,NaN,0.03,NaN,0.02
inflation,0.19,NaN,0.06,NaN,0.04,NaN,0.02


## 5. Portfolios

In [10]:
def valid_corr(R):
    """Nearest valid correlation matrix: negative eigenvalues set to a small positive number, unit diagonal."""
    vals, vecs = np.linalg.eigh(R)
    if vals.min() > 1e-8:
        return R
    R = vecs @ np.diag(np.clip(vals, 1e-8, None)) @ vecs.T
    d = np.sqrt(np.diag(R))
    return R / np.outer(d, d)


realised_matrix = {t: C[corr_months.get_loc(t + pd.offsets.MonthEnd(1))][0, 1] for t in decisions}
spy_tlt = forecasts.assign(oracle_spy_tlt=pd.Series(realised_matrix))
rows, repaired = [], 0
for t in decisions:
    vol_t = ret_d.loc[:t].tail(VOL_DAYS).std().to_numpy() * np.sqrt(252)
    base = rolling_matrix(t)
    for name in spy_tlt:
        R = base.copy()
        R[0, 1] = R[1, 0] = spy_tlt.loc[t, name]
        R_ok = valid_corr(R)
        repaired += not np.array_equal(R, R_ok)
        cov = np.outer(vol_t, vol_t) * R_ok
        w = helpers.erc_weights(cov)
        leverage = min(TARGET_VOL / np.sqrt(w @ cov @ w), MAX_LEVERAGE)
        rows.append({"date": t, "strategy": name, **dict(zip(ASSETS, leverage * w)), "leverage": leverage})
positions = pd.DataFrame(rows).set_index(["strategy", "date"]).sort_index()
print("matrices repaired to be valid:", repaired, "of", len(rows))

rf = load("riskfree_monthly")["rf"]
results = {name: helpers.backtest(positions.loc[name, ASSETS], ret_m, rf, COST_BPS) for name in spy_tlt}

# same portfolio as stage 4 when the forecast is the 12-month average
stage4 = load("portfolio_returns")
assert np.allclose(results["rolling_12m"]["net"], stage4["rp_rolling"].reindex(results["rolling_12m"].index), atol=1e-12)
print("rolling_12m portfolio identical to rp_rolling of stage 4")

matrices repaired to be valid: 2 of 2596
rolling_12m portfolio identical to rp_rolling of stage 4


In [11]:
NAMES = list(spy_tlt)
rows = {}
for period, (a, b) in PERIODS.items():
    rows[(period, "60_40 (stage 4)")] = helpers.performance(
        pd.DataFrame({"net": stage4["60_40"], "turnover": np.nan, "exposure": 1.0}).loc[a:b], rf)
    for name in NAMES:
        rows[(period, name)] = helpers.performance(results[name].loc[a:b], rf)
perf = pd.DataFrame(rows).T
perf.index.names = ["period", "strategy"]
perf.loc["full sample"].round(3)

,excess return,volatility,sharpe,max drawdown,worst month,turnover per year,average exposure,months
strategy,,,,,,,,
60_40 (stage 4),0.069,0.104,0.663,-0.285,-0.107,NaN,1.000,236.0
historical_mean,0.071,0.107,0.663,-0.181,-0.087,1.952,1.345,236.0
last_month,0.074,0.110,0.675,-0.184,-0.131,3.075,1.388,236.0
rolling_12m,0.073,0.107,0.684,-0.177,-0.101,1.978,1.358,236.0
ols_all,0.080,0.120,0.666,-0.208,-0.158,2.203,1.372,236.0
ridge_market,0.073,0.108,0.679,-0.172,-0.105,1.981,1.349,236.0
ridge_all,0.073,0.108,0.679,-0.171,-0.108,1.984,1.349,236.0
forest_market,0.075,0.108,0.693,-0.166,-0.101,2.061,1.363,236.0
forest_all,0.075,0.107,0.695,-0.166,-0.101,2.069,1.360,236.0


In [12]:
for period in PERIODS:
    if period != "full sample":
        print(f"\n{period}:")
        print(perf.loc[period, ["excess return", "volatility", "sharpe", "max drawdown"]].round(3).to_string())


2007-2016:
                 excess return  volatility  sharpe  max drawdown
strategy                                                        
60_40 (stage 4)          0.070       0.092   0.760        -0.285
historical_mean          0.060       0.101   0.594        -0.181
last_month               0.074       0.112   0.666        -0.184
rolling_12m              0.066       0.107   0.619        -0.177
ols_all                  0.083       0.130   0.636        -0.208
ridge_market             0.067       0.107   0.628        -0.172
ridge_all                0.068       0.107   0.632        -0.171
forest_market            0.069       0.107   0.646        -0.166
forest_all               0.069       0.107   0.650        -0.166
boosting_market          0.069       0.106   0.646        -0.173
boosting_all             0.068       0.106   0.644        -0.172
oracle_spy_tlt           0.078       0.109   0.720        -0.165

2017-2026:
                 excess return  volatility  sharpe  max drawdown
s

### Against `rp_rolling`, and how much of the perfect forecast is recovered

Sharpe difference against the 12-month average (Ledoit-Wolf test), and the share of the gain of `oracle_spy_tlt`
that each forecast recovers: (Sharpe of the model - Sharpe of rolling) / (Sharpe of the oracle - Sharpe of rolling).

In [13]:
excess = {name: results[name]["net"] - rf.reindex(results[name].index) for name in NAMES}
sharpe = perf.loc["full sample", "sharpe"]
gain = sharpe["oracle_spy_tlt"] - sharpe["rolling_12m"]
tests = {}
for name in NAMES:
    if name != "rolling_12m":
        tests[name] = {**helpers.sharpe_diff_test(excess[name], excess["rolling_12m"]),
                       "share of oracle gain": (sharpe[name] - sharpe["rolling_12m"]) / gain}
tests = pd.DataFrame(tests).T
tests.round(3)

,sharpe difference,se,p,share of oracle gain
historical_mean,-0.021,0.021,0.333,-0.200
last_month,-0.009,0.039,0.820,-0.087
ols_all,-0.018,0.036,0.613,-0.176
ridge_market,-0.004,0.012,0.710,-0.042
ridge_all,-0.005,0.014,0.701,-0.051
forest_market,0.009,0.012,0.456,0.085
forest_all,0.011,0.011,0.323,0.108
boosting_market,0.007,0.012,0.568,0.065
boosting_all,0.006,0.012,0.603,0.059
oracle_spy_tlt,0.103,0.026,0.000,1.000


## 6. Save

- `forecast_predictions.csv`: forecast of each model for month `t+1`, and the realised correlation
- `forecast_accuracy.csv`: the table of section 4, all periods
- `forecast_importance.csv`: average share of each feature
- `forecast_portfolio_returns.csv`: monthly net returns of the portfolios of section 5
- `forecast_performance.csv`: the performance table of section 5, all periods

In [14]:
predictions_out = spy_tlt.assign(realised_next_month=realised)
returns_out = pd.DataFrame({name: results[name]["net"] for name in NAMES}).rename_axis("date")
for df, name in [(predictions_out, "forecast_predictions"), (accuracy, "forecast_accuracy"),
                 (importance.rename_axis("feature"), "forecast_importance"),
                 (returns_out, "forecast_portfolio_returns"), (perf, "forecast_performance")]:
    path = helpers.save_processed(df, name)
    print(f"saved {path.name}: {df.shape[0]} rows x {df.shape[1]} columns")

saved forecast_predictions.csv: 236 rows x 12 columns
saved forecast_accuracy.csv: 60 rows x 6 columns
saved forecast_importance.csv: 11 rows x 7 columns
saved forecast_portfolio_returns.csv: 236 rows x 11 columns
saved forecast_performance.csv: 72 rows x 8 columns
